# <center> Astro-inference from A to Z <center>

In [ ]:
import os
SAVEDIR = '../Trash' #This is where everything will be saved to
os.makedirs(SAVEDIR, exist_ok = True)

In [ ]:
device = 'cuda' # this is the hardware used for computation. change to 'cpu' if you do not have a GPU

## Plotting settings

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
hist_settings = dict(
    bins = 40,
    histtype = 'step',
    lw = 3,
    density = True
)
import matplotlib.lines as mlines
import matplotlib.pyplot as plt

plt.style.use('default')
def figsize(scale, wc = 1, hc = 1):
    fig_width_pt = 513.17 #469.755                  # Get this from LaTeX using \the\textwidth
    inches_per_pt = 1.0/72.27                       # Convert pt to inch
    golden_mean = (np.sqrt(5.0)-1.0)/2.0            # Aesthetic ratio (you could change this)
    fig_width = fig_width_pt*inches_per_pt*scale    # width in inches
    fig_height = fig_width*golden_mean              # height in inches
    fig_size = [wc * fig_width,hc * fig_height]
    return fig_size
plt.rcParams.update(plt.rcParamsDefault)

params = {#'backend': 'pdf',
        'axes.labelsize': 12,
        'lines.markersize': 4,
        'font.size': 10,
        'xtick.major.size':6,
        "xtick.top": True,
        "ytick.right": True,
        "xtick.minor.visible": True,
        "xtick.major.top": True, 
        "xtick.minor.top": True,
        "ytick.minor.visible": True, 
        "ytick.major.right": True, 
        "ytick.minor.right": True,
        "ytick.direction": "in",
        "xtick.direction": "in",
        'xtick.minor.size':3,  
        'ytick.major.size':6,
        'ytick.minor.size':3, 
        'xtick.major.width':0.5,
        'ytick.major.width':0.5,
        'xtick.minor.width':0.5,
        'ytick.minor.width':0.5,
        'lines.markeredgewidth':1,
        'axes.linewidth':1.2,
        'legend.fontsize': 7,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'savefig.dpi':200,
        'path.simplify':True,
        'font.family': 'serif',
        'font.serif':'Times',
        # "text.usetex": True,
        #'text.latex.preamble': [r'\usepackage{amsmath}'],
        'text.usetex':False,
        'figure.figsize': figsize(0.5, 1, 1)}
plt.rcParams.update(params)
plt.rcParams['font.family'] = 'STIXGeneral'  # Closely matches Computer Modern
plt.rcParams['mathtext.fontset'] = 'stix'    # Use STIX for math

## Step 1: What are your GWB hyperparameters (binary evolution params)?

In [ ]:
import numpy as np
import json, glob, pickle, os
from scipy.stats import qmc
import corner
import random
from scipy.stats.distributions import norm, uniform

In [ ]:
# These are the 'phenom' binary evolution parameters' distributions
[
    'Uniformm(0.1, 11)',
    'Normal(-2.56, 0.4)',
    'Normal(10.9, 0.4)',
    'Normal(8.6, 0.2)',
    'Normal(0.32, 0.15)',
    'Uniform(-1.5, 0.0)'
]
n_astro_params = 6 # this could change depending on the model
crn_bins = 5 # the number of GWB frequency-bins

In [ ]:
astro_draws = int(1e4) # number of draws from the astro distributions (should be long enough to encode the uncertainty in astro-params)
sampler = qmc.LatinHypercube(d=6, strength=1).random(n=astro_draws)
sampler.shape

### Since having a high dimensional grid is not possible, we use Latin Hypercube Sampling

In [ ]:
lhd = []
lhd.append(uniform(loc = 0.1, scale = 11 - 0.1).ppf(sampler[:, 0])) 
lhd.append(norm(loc=-2.56, scale=0.4).ppf(sampler[:, 1])) 
lhd.append(norm(loc=10.9, scale=0.4).ppf(sampler[:, 2])) 
lhd.append(norm(loc=8.6, scale=0.2).ppf(sampler[:, 3])) 
lhd.append(norm(loc=0.32, scale=0.15).ppf(sampler[:, 4])) 
lhd.append(uniform(loc = -1.5, scale = 1.5).ppf(sampler[:, 5]))
lhd = np.array(lhd).T
lhd.shape

In [ ]:
labels = [r'$\tau_f$', r'$\phi_{0}$', r'$m_{\phi,0}$', r'$\mu$', r'$\epsilon_{\mu}$', r'$\nu_{\rm{inner}}$']
fig = plt.figure(figsize=figsize(0.5, 3, 4))
corner.corner(lhd, color='blue', fig = fig, bins=20, hist_bin_factor=2, data_kwargs={'ms':3}, hist_kwargs={'density': True, 'linewidth':2}, 
                    contour_kwargs={'linewidths':.0001}, labels = labels,show_titles = True,plot_contours = False,
            truth_color = 'black', desity = True, plot_datapoints = False)
plt.show()

In [ ]:
np.save(SAVEDIR + '/test_astro_params.npy', lhd) #save it for later use!

## Step 2: Ask holodeck for GWB spectrum

In [ ]:
import numpy as np
from holodeck import sams, utils, hardening, host_relations
from holodeck.constants import MSOL, YR, PC, GYR
from holodeck.librarian.lib_tools import _Param_Space, PD_Uniform, PD_Normal
from joblib import Parallel, delayed
import os
from natsort import natsorted
import glob
from tqdm.auto import trange
from tqdm_joblib import tqdm_joblib

In [ ]:
def init_sam(sam_shape, params):
    gsmf = sams.GSMF_Schechter(
        phi0=params['gsmf_phi0_log10'],
        phiz=params['gsmf_phiz'],
        mchar0_log10=params['gsmf_mchar0_log10'],
        mcharz=params['gsmf_mcharz'],
        alpha0=params['gsmf_alpha0'],
        alphaz=params['gsmf_alphaz'],
    )
    gpf = sams.GPF_Power_Law(
        frac_norm_allq=params['gpf_frac_norm_allq'],
        malpha=params['gpf_malpha'],
        qgamma=params['gpf_qgamma'],
        zbeta=params['gpf_zbeta'],
        max_frac=params['gpf_max_frac'],
    )
    gmt = sams.GMT_Power_Law(
        time_norm=params['gmt_norm']*GYR,
        malpha=params['gmt_malpha'],
        qgamma=params['gmt_qgamma'],
        zbeta=params['gmt_zbeta'],
    )
    mmbulge = host_relations.MMBulge_KH2013(
        mamp_log10=params['mmb_mamp_log10'],
        mplaw=params['mmb_plaw'],
        scatter_dex=params['mmb_scatter_dex'],
    )

    sam = sams.Semi_Analytic_Model(
        gsmf=gsmf, gpf=gpf, gmt=gmt, mmbulge=mmbulge,
        shape=sam_shape,
    )
    return sam

def init_hard(sam, params):
    hard = hardening.Fixed_Time_2PL_SAM(
        sam,
        params['hard_time']*GYR,
        sepa_init=params['hard_sepa_init']*PC,
        rchar=params['hard_rchar']*PC,
        gamma_inner=params['hard_gamma_inner'],
        gamma_outer=params['hard_gamma_outer'],
    )
    return hard


params = dict(
    hard_time=3.0,          #This will be varied
    hard_sepa_init=1e4,     
    hard_rchar=100.0,       
    hard_gamma_inner=-1.0, ##This will be varied
    hard_gamma_outer=+2.5,

    gsmf_phi0_log10=-2.77, ##This will be varied
    gsmf_phiz=-0.6,
    gsmf_mchar0_log10=11.24,##This will be varied
    gsmf_mcharz=0.11,
    gsmf_alpha0=-1.21,
    gsmf_alphaz=-0.03,

    gpf_frac_norm_allq=0.025,
    gpf_malpha=0.0,
    gpf_qgamma=0.0,
    gpf_zbeta=1.0,
    gpf_max_frac=1.0,

    gmt_norm=0.5,           
    gmt_malpha=0.0,
    gmt_qgamma=-1.0,      
    gmt_zbeta=-0.5,

    mmb_mamp_log10=8.69, ##This will be varied
    mmb_plaw=1.10,          
    mmb_scatter_dex=0.3, ##This will be varied
)

In [ ]:
SAM_SHAPE = (30, 30, 30) # number of grid points in the order of (total mass, mass ratio, red shift)
NUM_REALS = int(1e4)    # Number of realizations of GWB to generate (Poisson samples to account for uncertainty in the number of sources)
NUM_LOUDEST = 0   # Number of 'loudest' binaries to extract in each frequency bin (does not affect any calculation)
PTA_DUR_yr = 20
PTA_DUR = PTA_DUR_yr * YR
NUM_FREQS = 5 #this for GWB (1/T, 2/T, ..., NUM_FREQS/T)
freqs = np.arange(1/PTA_DUR, (NUM_FREQS+ .001)/PTA_DUR, 1/PTA_DUR)

### lets load the astro-params or use the ones from above

In [ ]:
theta_master = np.array(lhd)
# theta_master = np.load('../test_astro_params.npy', mmap_mode = 'r')

In [ ]:
def doit(rr):
    
    theta = theta_master[rr] #randomly selected draws from the distributions
    params['hard_time'] = theta[0]
    params['gsmf_phi0_log10'] = theta[1]
    params['gsmf_mchar0_log10'] = theta[2]
    params['mmb_mamp_log10'] = theta[3]
    params['mmb_scatter_dex'] = theta[4]
    params['hard_gamma_inner'] = theta[5]

    sam = init_sam(sam_shape = SAM_SHAPE, params = params)
    hard = init_hard(sam, params)
    fobs_gw_cents, fobs_gw_edges = utils.pta_freqs(PTA_DUR, NUM_FREQS)

    hc_ss_ph = sam.gwb_new(fobs_gw_edges, hard=hard, realize=NUM_REALS)

    spectrum = 0.5 * np.log10(hc_ss_ph**2/(12*np.pi**2 * freqs[:, None]**3 * PTA_DUR)) #for detection runs we use spectrum instead of strain!
    
    if np.isfinite(spectrum).all(): # no silly business
        np.save(SAVEDIR + f'/{rr}_{PTA_DUR_yr}yrs.npy', spectrum) # save the spectrum as a npy file (compressed and portable!)

In [ ]:
# Run them all in parallel
astro_draws_to_do = 20 #do the first 20 samples from the astro params
with tqdm_joblib(desc="Processing", total=astro_draws_to_do ) as progress_bar:
    Parallel(n_jobs=20)(delayed(doit)(i) for i in range(astro_draws_to_do ))

## Step 3: Prepare the data

In [ ]:
paths = natsorted(glob.glob(SAVEDIR + f'/*_{PTA_DUR_yr}yrs.npy')) #this is what you made just above
paths

### Combine all spectrum into one file which is more convenient (the file is memory mapped. do not worry about memory usage)

In [ ]:
one_file = np.lib.format.open_memmap(SAVEDIR + f'/gwb_spectrum_samples_{PTA_DUR_yr}yrs.npy', 
                    mode='w+', 
                    dtype='float64', 
                    shape=(len(paths), NUM_REALS , NUM_FREQS), 
                    fortran_order=False)

for idx in trange(len(paths)):
    one_file[idx] = np.load(paths[idx]).T # we wont use the `one_file`. it is just here to dump the spectrum into a single file

In [ ]:
one_file[-1]

### Now lets prepare the data for flow training

In [ ]:
gwb_data = np.load(SAVEDIR + f'/gwb_spectrum_samples_{PTA_DUR_yr}yrs.npy', mmap_mode='r')
par_data = theta_master[:gwb_data.shape[0]] # make sure the right astro samples are used. remove ':gwb_data.shape[0]' if needed

n_real = gwb_data.shape[1]
n_samp = gwb_data.shape[0]
n_pars = par_data.shape[-1]

par_data = np.broadcast_to(par_data, (n_real, n_samp, n_pars)).transpose((1, 2, 0)) #this just makes sure astro-params have the same shape as gwb spectrum
assert np.all(par_data[:, :, 100] == par_data[:, :, 89]) # this makes sure the astro params of two values of gwb spectrum is the same if they have to be

## The first axis is samples, the second is realization
par_data = par_data.transpose((0, 2, 1))
## The third axis combines `gwb_freq` params with other `params`. This is just a matter of organization!
chain = np.concatenate((par_data, gwb_data), axis = 2)

# bad_samp = np.load(f'/data/taylor_group/Nima/pandora/holodeck/hope/helper/nimsim_lib_gwb_{PTA_DUR_yr}yrs_{NUM_FREQS}bins_bad_samples_aggressive.npy')
# chain = np.delete(chain, bad_samp, axis = 0)
assert chain.all()

## Some things that we need to know about the library...
## Perform manual linear transformation of all the parameters (gwb + astro) to `[-B = -5, B = 5]`. The specific choice of `B` should not matter.
## There are other ways of doing this using `tanh` or deviding by standard deviation after subtracting the mean. The goal is to normalize the entire array
## to have the same range. To work wih machine learning algorithms, this is necessary
B = 5
min_x = np.min(chain, axis = (0, 1))
max_x = np.max(chain, axis = (0, 1))
mean = (max_x + min_x) / 2
half_range = (max_x - min_x) / 2
chain = B * (chain - mean) / half_range

In [ ]:
chain.max(), chain.min()

### Save the normalized samples so that we can use them for flow training later

In [ ]:
np.save(SAVEDIR + f'/gwb_spectrum_samples_{PTA_DUR_yr}yrs_normalized.npy', chain[..., n_pars:])
np.save(SAVEDIR + f'/ast_spectrum_samples_{PTA_DUR_yr}yrs_normalized.npy', chain[:, 0, :n_pars])
np.savez_compressed(SAVEDIR + f'/gwb_spectrum_samples_{PTA_DUR_yr}yrs_mapping_data.npy', 
                B = B, mean = mean, half_range = half_range)

## Step 4: Normalizing Flows Training

In [ ]:
import numpy as np
import random
import torch
import zuko
import os
import matplotlib.pyplot as plt
import corner
from tqdm.auto import trange
import cloudpickle as cpickle
import itertools
torch.set_default_dtype(torch.float64)
torch.set_default_device(device)

In [ ]:
PTA_DUR_yr = 20
NUM_FREQS = 5
n_pars = 6 # number of astro params


chain_rho = np.load(SAVEDIR + f'/gwb_spectrum_samples_{PTA_DUR_yr}yrs_normalized.npy', mmap_mode = 'r')
chain_ast = np.load(SAVEDIR + f'/ast_spectrum_samples_{PTA_DUR_yr}yrs_normalized.npy', mmap_mode = 'r')

total_sample_size_data = chain_rho.shape[1]
total_sample_size_cont = chain_ast.shape[0]

data = torch.tensor(chain_rho.reshape(total_sample_size_data * total_sample_size_cont, NUM_FREQS))
# There are more memory efficient ways to do this, but this is the simplest way to do it.
# The context is the astro params, which are repeated for each gwb spectrum sample in the data.
# This is a bit memory inefficient, but it works for small datasets.
# If you have a large dataset, you might want to use a more memory efficient way.
contx = torch.tensor(np.repeat(chain_ast[:, None], total_sample_size_data, axis = 1).reshape(data.shape[0], n_pars))

# Define the number of features and context features
num_features = data.shape[-1]
context_features = contx.shape[-1]

In [ ]:
# Below are the tuning parameters for the normalizing flows. You should try all of them to see which one works best for your data.
# The first is the batch size (how many samples per learning iteration)
# The second is the learning rate of the ADAM optimizer
# The third is the depth of the neural network

# all_pos = list(itertools.product([512, 256, 128, 64], [3e-3, 1e-3, 1e-4], [[512] * 2, [512] * 4, [512] * 6]))
all_pos = list(itertools.product([512], [1e-4], [[512] * 8]))

### Note: I use `zuko` for normalizing flows. JAX-based flows could be faster, but `zuko` is more capable of accurate emulation. If you want to use JAX-based flows, be sure to double check the quality of the flow. If you messs around with JAX-based flows, you can find a good settings.

In [ ]:
def doit(idx):
    one_pos = all_pos[idx]
    # Initialize the Neural Spline Flow
    flow = zuko.flows.spline.NSF(
        num_features,
        context_features,
        bins=8,  # Number of bins for the spline
        passes=2,  # Number of passes (2 for coupling)
        hidden_features = one_pos[-1]
    ).cuda() #######################################NOTE: remove `.cuda()` if you are running on cpu

    # Train the flow
    batch_size = one_pos[0]
    total_size = data.shape[0]

    print(data.shape)
    print(contx.shape)
    optimizer = torch.optim.Adam(flow.parameters(), lr=one_pos[1])

    for epoch in trange(int(1e4)): ## do this for int(1e5)
        optimizer.zero_grad()

        # rand_sample = random.randint(0, chain_ast.shape[1] - 1)
        rand_sample = random.sample(range(total_size), k = batch_size)

        # Obtain the distribution from the flow
        dist = flow(contx[rand_sample])
        # Compute the negative log-likelihood
        loss = -dist.log_prob(data[rand_sample]).mean()
        loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item()}")
    torch.save([flow], SAVEDIR + f"/zuko_flow.pkl")

for _ in range(1):
    doit(_)

## Step 5: Astro-inference

In [ ]:
import jax
import jax.numpy as jnp
import jax.scipy as jsp
import jax.random as jar
jax.config.update("jax_enable_x64", True)
import itertools

import numpy as np
from scipy.stats import norm
from scipy.stats import halfnorm

from pandora import models, GWBFunctions
from pandora import utils as putils
from pandora import LikelihoodCalculator as LC
from pandora import nf_dist
from joblib import Parallel, delayed

from enterprise_extensions.model_utils import get_tspan
from enterprise_extensions.models import model_general
from enterprise_extensions import blocks
from enterprise.signals import signal_base, gp_signals
import scipy.linalg as sl

import pickle, json, os, corner, glob, random, copy, time, inspect, math, sys
from natsort import natsorted
from tqdm import tqdm
import torch
torch.set_default_dtype(torch.float64)


### Make sure the settings below match the settings above!

In [ ]:
crn_bins = 5
gwb_freq_bins = crn_bins

n_pars = 6
Tspan = 20 * 365.25 * 86400
freqs = np.arange(1/Tspan, (crn_bins + .001)/Tspan, 1/Tspan)

## This is a wrapper for `zuko`'s flow to use in Bayesian inferrence. You need to write your own wrapper if you want to use a different package

In [ ]:
aux_info = np.load(SAVEDIR + '/gwb_spectrum_samples_20yrs_mapping_data.npy.npz')
B = aux_info['B']
mean = aux_info['mean']
half_range = aux_info['half_range']

nf = torch.load(SAVEDIR + f"/zuko_flow.pkl",
                                weights_only = False,
                                map_location = device)[0]

nf_dist_object = nf_dist.NFastroinference(nf_object = nf,
        nf_type = 'rho|theta',
        nf_object_device = device,
        mean = np.array(mean),
        half_range = np.array(half_range),
        scale = B,
        rho_idxs = np.array(range(n_pars, n_pars + gwb_freq_bins), dtype = int),
        ast_param_idxs = np.array(range(n_pars), dtype = int))

### Build the bases and the white noise covariance matrix using enterprise. (JAX SHOULD NOT be used here due to memory concerns; hence, the reliance on `enterprise`)

### if you have a featherfile:

In [ ]:
from enterprise.pulsar import FeatherPulsar
import glob

feather_files = glob.glob('/data/taylor_group/NANOGrav/NG20_v1p1/ng20_v1p1_dmx_feathers/*.feather')

psrs = []
for feather_file in feather_files:
    psr = FeatherPulsar.read_feather(feather_file)
    psrs.append(psr)

### if you have a pickle file:

In [ ]:
# import pickle
# with open('./your_directory/.pickle', 'rb') as fin:
#     psrs = pickle.load(fin)

### PTA White Noise noise dictionary:

In [ ]:
import json
with open('/data/taylor_group/NANOGrav/NG20_v1p1/ng20_v1p1_dmx_noise_dict.json', 'r') as fin:
    noise_dict = json.load(fin)

# psrlist = [psr.name for psr in psrs]
# noise_dict = {}
# for pname in psrlist:
#     noise_dict.update({pname + '_efac': 1.0})
#     noise_dict.update({pname + '_log10_t2equad': -np.inf})

In [ ]:
tm = gp_signals.MarginalizingTimingModel(use_svd=True)
wn = blocks.white_noise_block(
    vary=False,
    inc_ecorr=True, #if you are dealing with real data, set this to `True`
    gp_ecorr=False,
    select='backend', #if you are dealing with real data, set this to `backend`
    tnequad=False,
)
# These do not matter do not change them!
rn = blocks.red_noise_block(
    psd="powerlaw",
    prior="log-uniform",
    Tspan=Tspan,
    components=crn_bins,
    gamma_val=None,
)
# These do not matter do not change them!
gwb = blocks.common_red_noise_block(
    psd="spectrum",
    prior="log-uniform",
    # modes = freqs,
    Tspan = Tspan,
    components=crn_bins,
    gamma_val=None,
    name="gw",
    orf="hd",
)
s = tm + wn + rn + gwb

pta = signal_base.PTA(
    [s(p) for p in psrs], signal_base.LogLikelihoodDenseCholesky
)
pta.set_default_params(noise_dict)

TNr = np.concatenate(pta.get_TNr(params={}))
TNT = np.array(sl.block_diag(*pta.get_TNT(params={})))
print(f'******{TNT.shape}*********')

### For astro-inferences, TNr and TNT are the data! Do not worry if you do not know what they are. You can think about them in terms of the projection of the timing data in the frequency space (very crudely speaking!)

### Use `pandora` to compute the likelihood

In [ ]:
lower_gwb_psd_value_in_log10 = -12.
upper_gwb_psd_value_in_log10 = -4.

### Here you need to choose the noise model. `hd_spectrum` for gwb plus varied gamma powerlaw for non-gwb noise is the standard 

In [ ]:
chosen_psd_model, chosen_orf_model, gwb_helper_dictionary = putils.hd_spectrum(renorm_const=1,crn_bins=crn_bins,
                                                                            lower_halflog10_rho=lower_gwb_psd_value_in_log10, 
                                                                            upper_halflog10_rho=upper_gwb_psd_value_in_log10)
gwb_helper_dictionary

### Here, we make the actual noise model

In [ ]:
o = models.UniformPrior(gwb_psd_func = chosen_psd_model,
                orf_func = chosen_orf_model,
                crn_bins = crn_bins,
                int_bins = crn_bins,
                f_common = freqs, 
                f_intrin = freqs,
                df = 1/Tspan,
                Tspan = Tspan, 
                Npulsars = len(psrs),
                psr_pos = [psr.pos for psr in psrs],
                gwb_helper_dictionary = gwb_helper_dictionary,
                renorm_const = 1)

### This step constructs the prior probability for the astro parameters. Since they are analytic, it is easy!

In [ ]:
mean_astro = np.array([5.55, -2.56, 10.9, 8.6, 0.32, -0.75])
std_astro = np.array([100, 0.4, 0.4, 0.2, 0.15, 100])
astr_prior_lower_lim = np.load(SAVEDIR + '/test_astro_params.npy', mmap_mode='r').min(axis = 0)
astr_prior_upper_lim = np.load(SAVEDIR + '/test_astro_params.npy', mmap_mode='r').max(axis = 0)
def astro_additional_prior_func_normal(xs):
    '''
    this function is for the inclusion of the normal priors. You can change it to a multi-variate normal if needed.
    just choose your mean and std for the astro parameters. BE CAREFUL! The order of `xs` is the same order as your 
    trained normalizing flow object.

    :param xs: an array of length equal to the number of astro params
    '''
    return norm.logpdf(xs, loc=mean_astro, scale=std_astro).sum()

In [ ]:
astr_prior_lower_lim, astr_prior_upper_lim

### Here, we make the likelihood calculator class

In [ ]:
m = LC.AstroInferenceModel(nf_dist = nf_dist_object,
        num_astro_params = n_pars,
        astr_prior_lower_lim = astr_prior_lower_lim,
        astr_prior_upper_lim = astr_prior_upper_lim,
        astro_additional_prior_func = astro_additional_prior_func_normal,
        run_type_object = o,
        psrs = None,
        device_to_run_likelihood_on = device,
        astro_param_fixed_values = np.array([False]), 
        astro_param_fixed_indices = np.array([False]), 
        fixed_spectrum = np.array([False]),
        TNr=jnp.array(TNr),
        TNT=jnp.array(TNT),
        
        ### If you have TNr and TNT the settings below does not matter!
        noise_dict=None,
        backend="none",
        tnequad=False,
        inc_ecorr=False,
        del_pta_after_init=True,
        matrix_stabilization=False,
        delta = 1e-6,)

In [ ]:
for _ in range(10000, 10000 + 10):
    x0 = m.make_initial_guess(seed = _)
    ans = m.get_lnliklihood(x0)
    if np.isfinite(ans):
        print(_, ans)
        break
    else:
        assert np.isfinite(ans)

In [ ]:
%timeit m.get_lnliklihood(x0)

### Sample it!

In [ ]:
m.sample(
            x0 = np.array(x0),
            niter = int(1e6),
            savedir = SAVEDIR,
            resume=True,
            seed=None,
            include_groups = True,
            include_IRN_groups = False)

## If you want to use an HMC sampler, you can! Use `m.get_lnliklihood_non_astro` as the likelihood function plus your jax flow logrob plus the the JAXified version of `astro_additional_prior_func_normal`. I just do not trust JAX-based flows, yet!